# DisorderNet Quick Screen — breakthrough go/no-go

**Purpose:** Estimate whether the **ultra** paradigm (ESM-2 650M + LoRA + v6 ensemble) can reach **≥0.90 AUC** *before* committing to an 18–24h full CV.

| Mode | Proteins | Folds | Train profile | ~Time (A100-40GB / 80GB) | Use when |
|---|---:|---:|---|---|---|
| `flash` | 120 | 2 | toy `screen` (CNN) | 45–90 min / 20–40 min | Coarse sanity only |
| **`standard`** | 250 | 3 | **`screen_plus` (mini-ultra)** | 2–3 h / 40–90 min | **Recommended go/no-go** |
| `paradigm` | 350 | 3 | `screen_plus` | 4–6 h / ~1–3 h | Highest 650M fidelity before full CV |

**Important:** `standard` trains with the mini-ultra stack (SOTA head, rich features, FFN LoRA, homology splits)—not the old toy CNN recipe—so STOP/MODERATE reflects the real ultra path.

**Output:** `quick_screen_report.json` with tier `HIGH` / `MODERATE` / `LOW` / `STOP` and a **mode-aware** projected full-CV AUC band.

If tier is **HIGH** or **MODERATE (proceed=yes)** → run `DisorderNet_Colab_Pro.ipynb` with `QUALITY_PROFILE="ultra"`.


In [ ]:
# CELL 1 — Clone / path setup
import os, sys, subprocess
REPO_URL = "https://github.com/Tommaso-R-Marena/DisorderNet.git"
REPO_DIR = "DisorderNet"
if os.path.isfile("colab/disordernet_gpu.py"):
    sys.path.insert(0, ".")
elif os.path.isdir(REPO_DIR):
    sys.path.insert(0, REPO_DIR)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    sys.path.insert(0, REPO_DIR)
print("Repo ready.")

In [ ]:
# CELL 2 — Environment + screen mode
from colab.disordernet_gpu import (
    TrainConfig, install_dependencies, setup_environment,
    fetch_disprot, process_disprot, print_dataset_summary, load_esm_model,
)
from colab.quick_screen import (
    SCREEN_MODES, run_paradigm_quick_screen,
    print_quick_screen_report, save_quick_screen_report,
)

install_dependencies(quiet=True)

# flash (toy) | standard (recommended mini-ultra) | paradigm (larger mini-ultra)
SCREEN_MODE = "standard"
SCREEN_BACKBONE = "650M"  # "3B" for paradigm test on A100 40GB+
SEED = 42
# After code updates: Runtime → Restart session, then re-run cells so git pull picks up screen_plus.

cfg = TrainConfig.from_profile("screen_plus", seed=SEED)  # env probe; real profile comes from SCREEN_MODE
cfg = setup_environment(cfg)
print(f"Screen mode: {SCREEN_MODE} — {SCREEN_MODES[SCREEN_MODE]['description']}")


In [ ]:
# CELL 3 — Load DisProt (optional Drive mount)
MOUNT_DRIVE = True
if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        cfg.data_cache = "/content/drive/MyDrive/disordernet/disprot_raw.json"
        cfg.checkpoint_dir = "/content/drive/MyDrive/disordernet/checkpoints"
    except ImportError:
        MOUNT_DRIVE = False

entries = fetch_disprot(cache_path=cfg.data_cache)
proteins, skipped = process_disprot(entries, cfg)
print_dataset_summary(proteins, skipped)

In [ ]:
# CELL 4 — Load ESM-2 backbone
from colab.esm_backbone import get_backbone_spec, print_backbone_playbook
bb = getattr(cfg, 'esm_backbone', SCREEN_BACKBONE if 'SCREEN_BACKBONE' in dir() else '650M')
model_esm, alphabet, batch_converter = load_esm_model(cfg.device, backbone=bb)
cfg.esm_embed_dim = get_backbone_spec(bb).embed_dim
print_backbone_playbook(cfg.vram_gb)
print("ESM-2 ready.")


In [ ]:
# CELL 5 — Run quick screen (GPU CV + v6 ensemble + verdict)
report = run_paradigm_quick_screen(
    proteins=proteins,
    esm_backbone=model_esm,
    batch_converter=batch_converter,
    cfg=cfg,
    mode=SCREEN_MODE,
    seed=SEED,
    backbone=SCREEN_BACKBONE,
    run_ensemble=True,
    use_v6_pro=True,
)
print_quick_screen_report(report)
save_quick_screen_report(report, "quick_screen_report.json")

if report["verdict"]["proceed_full_ultra"]:
    print("\n✅ Recommendation: proceed with full DisorderNet_Colab_Pro.ipynb (ultra profile)")
else:
    print("\n⚠ Recommendation: do NOT start full 24h CV yet — tune paradigm or run paradigm mode first")